In [ ]:
import os
import uuid
from typing import TypedDict, Annotated

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_core.runnables import RunnableConfig
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import MemorySaver
from langgraph.store.base import BaseStore

os.environ["GOOGLE_API_KEY"] = "AQ.Ab8RN6KpAyxFD2hwogCwLZ1lWbnGAUq-TRVrY04vcBI0ifX88g"

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

store = InMemoryStore()
checkpointer = MemorySaver()

class State(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


def chatbot_node(state: State, config: RunnableConfig, *, store: BaseStore):
    user_id = config.get("configurable", {}).get("user_id", "default_user")
    namespace = ("memories", user_id)
    

    memories = store.search(namespace)
    memory_list = [m.value["data"] for m in memories] if memories else []
    memory_text = "\n".join([f"- {fact}" for fact in memory_list]) if memory_list else "No past memory found."
    
    system_prompt = (
        "You are an intelligent AI Assistant with long-term memory capabilities.\n"
        f"Here is what you currently know about this user from previous sessions:\n{memory_text}\n\n"
        "Use this contextual information to personalize your response."
    )
    
    formatted_messages = [SystemMessage(content=system_prompt)] + state["messages"]
    response = model.invoke(formatted_messages)
    

    user_last_msg = state["messages"][-1].content
    extraction_prompt = (
        "Extract any important new personal details, preferences, or facts about the user from this message.\n"
        f"Message: {user_last_msg}\n"
        "Return only key factual statements separated by newlines, or 'NONE' if no new information is present."
    )
    extracted_data = model.invoke(extraction_prompt).content.strip()
    
    if extracted_data and "NONE" not in extracted_data:
        for fact in extracted_data.split("\n"):
            if fact.strip():
                memory_id = str(uuid.uuid4())
                store.put(namespace, memory_id, {"data": fact.strip()})
    
    return {"messages": [response]}


builder = StateGraph(State)
builder.add_node("chatbot", chatbot_node)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)
app = builder.compile(checkpointer=checkpointer, store=store)


config_1 = {"configurable": {"thread_id": "1", "user_id": "ALI_123"}}
input_1 = {"messages": [HumanMessage(content="Hi, my name is ALI and I teach AI on YouTube.")]}

for chunk in app.stream(input_1, config=config_1, stream_mode="updates"):
    for node_name, node_output in chunk.items():
        if "messages" in node_output:
            print("Assistant:", node_output["messages"][-1].content)

    

config_2 = {"configurable": {"thread_id": "2", "user_id": "ALI_123"}}
input_2 = {"messages": [HumanMessage(content="Do you remember what I do for a living?")]}

print("--- Thread 2 Output ---")
for chunk in app.stream(input_2, config=config_2, stream_mode="updates"):
    for node_name, node_output in chunk.items():
        if "messages" in node_output:
            print("Assistant:", node_output["messages"][-1].content)


Assistant: Hi ALI, it's great to meet you! Teaching AI on YouTube sounds like a fantastic way to share knowledge. What kind of AI topics do you usually cover?
--- Thread 2 Output ---
Assistant: Yes, I do, Ali! You teach AI, and you even teach AI on YouTube.
